# Notebook 06: RAG Pipeline

In this notebook, we will build the complete **RAG (Retrieval-Augmented Generation)** pipeline.

## What You Will Learn

- How RAG works end-to-end
- How to retrieve relevant chunks
- How to send context to an LLM
- How to generate answers from retrieved context
- How to use LangChain's Retriever

## The RAG Pipeline

RAG has two main phases:

### Phase 1: Retrieval
```
User Question -> Embed Question -> Search Vector Store -> Get Relevant Chunks
```

### Phase 2: Generation
```
Question + Chunks -> Build Prompt -> Send to LLM -> Get Answer
```

### Combined:
```
User Question -> Retrieve Chunks -> Build Context -> LLM Generates Answer
```

## Why RAG?

Without RAG, LLMs can:
- Hallucinate (make up facts)
- Don't know about your documents
- Can't cite sources

With RAG, LLMs can:
- Answer from YOUR documents
- Cite exact sources
- Stay up-to-date with your data

## Step 1: Import Required Libraries

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
import os

print("All imports successful!")

C:\Users\Ahmed\AppData\Local\Temp\ipykernel_6580\3491207093.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All imports successful!


## Step 2: Setup the Vector Store

We'll reuse the code from previous notebooks to load the PDF and create the vector store.

In [2]:
# Load and split the PDF
loader = PyPDFLoader("../data/sample.pdf")
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(pages)

print(f"Loaded {len(pages)} pages, created {len(chunks)} chunks")

# Create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Create vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../chroma_db"
)

print("Vector store created!")

Loaded 4 pages, created 20 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store created!


## Step 3: Create a Retriever

A **retriever** is a LangChain object that wraps the vector store and returns relevant documents for a query.

In [3]:
# Create a retriever from the vector store
# This converts our vector store into a retriever object
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# The retriever can now find relevant documents for any question
print("Retriever created!")
print(f"Will return top 4 most relevant chunks")

Retriever created!
Will return top 4 most relevant chunks


## Step 4: Test the Retriever

In [4]:
# Let's test the retriever with a sample question
question = "What is the main topic of this document?"

# Get relevant documents
relevant_docs = retriever.invoke(question)

print(f"Question: {question}")
print(f"\nRetrieved {len(relevant_docs)} relevant chunks:\n")

for i, doc in enumerate(relevant_docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Page: {doc.metadata.get('page', 'unknown')}")
    print(f"Content: {doc.page_content}")
    print()

Question: What is the main topic of this document?

Retrieved 4 relevant chunks:

--- Chunk 1 ---
Page: 2
Content: Introduction to Artificial Intelligence
Chapter 4: Natural Language Processing
Natural Language Processing (NLP) is a field of AI that focuses on the interaction between computers and
human language. NLP enables machines to read, understand, and generate human language, making it
possible to build applications like chatbots, translation systems, and text summarizers.
Key NLP tasks include:
- Tokenization: Splitting text into individual words or subwords.

--- Chunk 2 ---
Page: 2
Content: Introduction to Artificial Intelligence
Chapter 4: Natural Language Processing
Natural Language Processing (NLP) is a field of AI that focuses on the interaction between computers and
human language. NLP enables machines to read, understand, and generate human language, making it
possible to build applications like chatbots, translation systems, and text summarizers.
Key NLP tasks include:

## Step 5: Create the LLM

We'll use the **Mistral** model running locally via Ollama.

In [5]:
# Create the Ollama LLM
llm = OllamaLLM(
    model="mistral",
    temperature=0.1,  # Low temperature = more focused answers
    num_ctx=4096      # Context window size
)

# Test the LLM with a simple question
test_response = llm.invoke("What is 2 + 2?")
print(f"LLM test response: {test_response}")

LLM test response:  The sum of 2 and 2 is 4.


## Step 6: Create the Prompt Template

This is the most important part! We create a prompt that:
1. Includes the retrieved context
2. Asks the model to answer from the context only
3. Tells the model what to do if the answer isn't found

In [6]:
# Create a prompt template
# {context} will be replaced with retrieved document chunks
# {question} will be replaced with the user's question

rag_prompt = PromptTemplate.from_template("""
You are a helpful AI assistant that answers questions based ONLY on the provided context.

## Context:
{context}

## Question:
{question}

## Instructions:
- Answer the question using ONLY the information from the context above.
- If the answer is not in the context, say: "I'm sorry, but I cannot find the answer to this question in the provided document."
- Be concise and accurate.
- Do not make up information or hallucinate.

## Answer:
""")

print("Prompt template created!")

Prompt template created!


## Step 7: Format the Retrieved Context

We need to combine all retrieved chunks into a single context string.

In [7]:
def format_docs(docs):
    """
    Takes a list of Document objects and combines them into a single string.
    Each chunk is numbered and separated by newlines.
    """
    formatted = "\n\n".join([
        f"[{i+1}] (Page {doc.metadata.get('page', '?')})\n{doc.page_content}"
        for i, doc in enumerate(docs)
    ])
    return formatted

# Test the formatter
test_docs = retriever.invoke("What is this document about?")
formatted_context = format_docs(test_docs)
print("Formatted context:")
print(formatted_context[:500])

Formatted context:
[1] (Page 2)
Introduction to Artificial Intelligence
Chapter 4: Natural Language Processing
Natural Language Processing (NLP) is a field of AI that focuses on the interaction between computers and
human language. NLP enables machines to read, understand, and generate human language, making it
possible to build applications like chatbots, translation systems, and text summarizers.
Key NLP tasks include:
- Tokenization: Splitting text into individual words or subwords.

[2] (Page 2)
Introduction t


## Step 8: Build the Complete RAG Chain

We'll use LangChain's **Runnable** pattern to chain everything together.

In [8]:
# Build the RAG pipeline using LangChain's chain syntax
# This chain:
# 1. Takes a question
# 2. Retrieves relevant documents
# 3. Formats them into context
# 4. Sends everything to the LLM with the prompt

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
)

print("RAG chain built successfully!")

RAG chain built successfully!


## Step 9: Test the Complete Pipeline

In [9]:
# Test the complete RAG pipeline
question = "What is the main topic discussed in this document?"

answer = rag_chain.invoke(question)

print(f"Question: {question}")
print(f"\nAnswer: {answer}")

Question: What is the main topic discussed in this document?

Answer:  The main topic discussed in this document is Natural Language Processing (NLP), a field of Artificial Intelligence that focuses on the interaction between computers and human language.


## Step 10: Test Multiple Questions

In [10]:
questions = [
    "What is the main topic of this document?",
    "What are the key points mentioned?",
    "When was this document written?",
]

for q in questions:
    print(f"\n{'=' * 60}")
    print(f"Question: {q}")
    print(f"{'=' * 60}")
    answer = rag_chain.invoke(q)
    print(f"Answer: {answer}")


Question: What is the main topic of this document?
Answer:  The main topic of this document is Natural Language Processing (NLP), a field of Artificial Intelligence that focuses on the interaction between computers and human language.

Question: What are the key points mentioned?
Answer:  The key points mentioned are:
1. Key NLP tasks: Tokenization, Sentiment Analysis, Named Entity Recognition (NER), Text Summarization, Machine Translation.
2. Application of AI in navigation for safe driving, entertainment industry (recommending movies/music, generating artwork/music, powering game characters).
3. Ethical concerns related to AI: Bias in AI systems leading to unfair outcomes.

Question: When was this document written?
Answer:  I'm sorry, but the provided document does not contain any information that can help determine when it was written.


## Step 11: Test with an Unanswerable Question

This tests our hallucination prevention.

In [11]:
# Ask something NOT in the document
question = "What is the capital of France?"

answer = rag_chain.invoke(question)

print(f"Question: {question}")
print(f"\nAnswer: {answer}")
print("\n(Note: The model should say it cannot find the answer in the document)")

Question: What is the capital of France?

Answer:  I'm sorry, but I cannot find the answer to this question in the provided document as it does not contain any information about the capital of France.

(Note: The model should say it cannot find the answer in the document)


## Key Takeaways

1. **RAG** = Retrieval + Augmentation + Generation
2. **Retriever** finds relevant chunks from the vector store
3. **Prompt Template** tells the LLM how to use the context
4. **Chain** connects retrieval and generation
5. We can prevent hallucinations by instructing the model to use only the provided context

## The Complete RAG Flow

```
User Question
    |
    v
[1] Embed Question
    |
    v
[2] Search Vector Store (ChromaDB)
    |
    v
[3] Get Top-K Relevant Chunks
    |
    v
[4] Format Chunks into Context
    |
    v
[5] Build Prompt (Context + Question + Instructions)
    |
    v
[6] Send to Mistral via Ollama
    |
    v
[7] Return Answer
```

## Next Steps

Proceed to **Notebook 07: Prompt Engineering** to improve our prompts.